# 06 Complexity Analysis

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import GroupKFold, KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import shap
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

df = pd.read_csv('../dataset/pd_speech_features.csv', header=1)
X = df.drop(['id', 'class'], axis=1)
y = df['class']
groups = df['id']


In [2]:
import sys
sys.path.append('../src')
import time
import pickle
import os
import xgboost as xgb
import lightgbm as lgb
import catboost as cb
import torch
from deep_tabular_model import DeepTabularModel

# Pick one single fold
gkf = GroupKFold(n_splits=5)
train_idx, test_idx = next(gkf.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# Preprocess
vt = VarianceThreshold(threshold=0.01)
X_train = vt.fit_transform(X_train)
X_test = vt.transform(X_test)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

skb = SelectKBest(f_classif, k=50)
X_train = skb.fit_transform(X_train, y_train)
X_test = skb.transform(X_test)

smote = SMOTE(random_state=RANDOM_STATE)
X_train, y_train = smote.fit_resample(X_train, y_train)

models = {
    'XGBoost': xgb.XGBClassifier(objective='binary:logistic', eval_metric='logloss', random_state=RANDOM_STATE),
    'LightGBM': lgb.LGBMClassifier(verbose=-1, random_state=RANDOM_STATE),
    'CatBoost': cb.CatBoostClassifier(verbose=0, random_state=RANDOM_STATE)
}

results = []

for name, model in models.items():
    # Train Time
    t0 = time.time()
    model.fit(X_train, y_train)
    t_train = time.time() - t0
    
    # Inference Latency per sample (ms)
    t0 = time.time()
    for _ in range(50):
        model.predict(X_test)
    t_inf = ((time.time() - t0) / 50 / len(X_test)) * 1000
    
    # Size
    with open('tmp_model.pkl', 'wb') as f:
        pickle.dump(model, f)
    size_kb = os.path.getsize('tmp_model.pkl') / 1024
    
    # Complexity
    if name == 'XGBoost': n_param = model.get_booster().trees_to_dataframe()['Tree'].max() + 1
    elif name == 'LightGBM': n_param = model.booster_.num_trees()
    elif name == 'CatBoost': n_param = model.tree_count_
    
    results.append({'Model': name, 'Train_Time_s': t_train, 'Inf_Latency_ms': t_inf, 'Size_KB': size_kb, 'Complexity': n_param})

# Deep Model
device = torch.device('cpu')
X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.FloatTensor(y_train.values)
X_test_t = torch.FloatTensor(X_test)

model = DeepTabularModel()
criterion = torch.nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

t0 = time.time()
model.train()
for _ in range(60):
    optimizer.zero_grad()
    logits = model(X_train_t)
    loss = criterion(logits, y_train_t)
    loss.backward()
    optimizer.step()
t_train = time.time() - t0

model.eval()
t0 = time.time()
with torch.no_grad():
    for _ in range(50):
        model(X_test_t)
t_inf = ((time.time() - t0) / 50 / len(X_test)) * 1000

torch.save(model.state_dict(), 'tmp_model.pt')
size_kb = os.path.getsize('tmp_model.pt') / 1024

n_param = sum(p.numel() for p in model.parameters() if p.requires_grad)

results.append({'Model': 'Deep Tabular', 'Train_Time_s': t_train, 'Inf_Latency_ms': t_inf, 'Size_KB': size_kb, 'Complexity': n_param})

res_df = pd.DataFrame(results)
print(res_df)
res_df.to_csv('../results/complexity_results.csv', index=False)


          Model  Train_Time_s  Inf_Latency_ms      Size_KB  Complexity
0       XGBoost      0.467601        0.006176   147.211914         100
1      LightGBM      0.350213        0.015565   343.874023         100
2      CatBoost     14.478331        0.009728  1108.122070        1000
3  Deep Tabular     63.644378        0.158568   121.885742       20417
